### Processamento e Limpeza dos dados (ETL)

In [ ]:
import pandas as pd
import json
import os
from datetime import datetime
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.processing.text_cleaning import clean_text

output_file = f"../data/raw/movies_raw_{datetime.now().strftime("%Y-%m-%d")}.json"
df = pd.read_json(output_file)

df.head(3)

In [ ]:
## Informações sobre as colunas
df.info()

### Seleção de colunas relevantes

In [ ]:
columns = [
    "id",
    "title",
    "overview",
    "genre_ids",
    "popularity",
    "vote_average",
    "release_date",
    "adult"
]

df = df[columns]
df.info()

### Tratamento de possíveis valores nulos

In [ ]:
df = df.dropna(subset=["title", "overview", "release_date"])

df.info()

### Mapeamento de gêneros 
A API retorna os gêneros dos filmes através de IDs, e precisamos transformar isso em nomes antes de entregar como fonte para o treinamento do modelo. E para isso foi feita a extração dos nomes dos gêneros através da 'genres_client.py'

In [ ]:
GENRES_PATH = f"../data/raw/genres_{datetime.now().strftime("%Y-%m-%d")}.json"

def load_genres_map():
    with open(GENRES_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    return {
        int(k): v
        for k, v in data.items()
    }

genre_map = load_genres_map()
genre_map

In [ ]:
df["genres"] = df["genre_ids"].apply(
    lambda ids: [genre_map.get(genre_id, "Unknown") for genre_id in ids] 
)

df.head()

### Normalização textual

In [ ]:
df["overview"] = df["overview"].apply(clean_text)

df["overview"]

### Criação de coluna combinada
Criação de coluna para virar base de preparação do modelo, servindo como base do vetor.

In [ ]:
df["content"] = df["overview"] + " " + df["genres"].apply(lambda x: " ".join([g.lower() for g in x]) if isinstance(x, list) else "")
df["content"][0]

### Remoção de duplicatas


In [ ]:
df = df.drop_duplicates(subset="id")
df.info()

### Correção de tipos

In [ ]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

df["release_date"].sample(5)